In [2]:
import requests
from bs4 import BeautifulSoup 
import numpy as np
import pandas as pd
from itertools import zip_longest
from urllib.request import urlopen
import selenium
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException
from tqdm import tqdm
import time

In [3]:
DRIVER_PATH = 'C:/Users/*USERNAME*/chromedriver/chromedriver.exe'

In [4]:
options = webdriver.ChromeOptions()
options.add_argument('ignore-certificate-errors')
options.add_argument("--disable-web-security")
options.add_argument("--allow-running-insecure-content")
options.headless = True

# Preventing Chrome from blocking mixed content
options.add_argument("--disable-features=BlockInsecurePrivateNetworkRequests")

options.add_argument("--window-size=1920,1200")
driver = webdriver.Chrome()

In [5]:
# Load the DataFrame
df = pd.read_excel("Scrape URLs.xlsx")
department_mapping = df.set_index("Code")["Department"].to_dict()
discipline_mapping = df.set_index("Code")["Discipline"].to_dict()

In [6]:
shanghai_result = pd.DataFrame(columns=['year', 'discipline', 'department', 'world_rank', 'institution', 
                                        'country_name', 'total_score', 'Q1', 'CNCI', 'IC', 'TOP', 'AWARD_SCORE'])

categories = ["Q1", "CNCI", "IC", "TOP", "AWARD"]

for j in tqdm(range(2023, 2024)):  # Adjust year
    print(f"\nProcessing year: {j}")

    for code in df[(df["Code"] >= 1)]["Code"]:
        url = f'https://www.shanghairanking.com/rankings/gras/{j}/RS0{code}'
        department = department_mapping.get(code, None)
        discipline = discipline_mapping.get(code, None)

        print(f"\nFetching data for discipline: {discipline}, department: {department}, URL: {url}")

        response = requests.get(url)
        if response.status_code == 404:
            print(f"Skipping {url} - Page not found (404)")
            continue  

        try:
            driver.get(url)
            soup = BeautifulSoup(driver.page_source, 'lxml')
            page_number = int(soup.find_all('li', class_='ant-pagination-item')[-1].get('class')[-1].split('-')[-1])
            print(f"Found {page_number} pages for {discipline}")
        except NoSuchElementException:
            print(f"Skipping {url} - Page structure issue")
            continue  

        universities_final = []
        countries_final = []
        total_score_final = []
        award_final = []
        world_number_final = []
        
        # Initialize a dictionary to store category scores
        category_scores_dict = {category: [] for category in categories}

        for i in range(2, page_number + 2):
            print(f"Processing page {i-1}/{page_number} for {department}")

            universities_results, countries_results, world_number_list = [], [], []
            total_score_results, award_results = [], []
            temp_category_scores = {category: [] for category in categories}

            for category in categories:
                print(f"\nSelecting category: {category}")

                try:
                    # Select the correct dropdown
                    dropdowns = WebDriverWait(driver, 2).until(
                        EC.presence_of_all_elements_located((By.CLASS_NAME, "inputWrapper"))
                    )
                    if len(dropdowns) < 4:
                        print(f"Failed to find the score dropdown for {category}")
                        continue

                    dropdown_element = dropdowns[3]
                    driver.execute_script("arguments[0].scrollIntoView(true);", dropdown_element)
                    driver.execute_script("arguments[0].click();", dropdown_element)
                    time.sleep(0.1)

                    # Select category
                    category_option = WebDriverWait(driver, 2).until(
                        EC.element_to_be_clickable((By.XPATH, f"//ul[@class='options']/li[contains(text(), '{category}')]"))
                    )
                    driver.execute_script("arguments[0].scrollIntoView(true);", category_option)
                    driver.execute_script("arguments[0].click();", category_option)
                    time.sleep(0.1)  

                    soup = BeautifulSoup(driver.page_source, 'lxml')

                except Exception as e:
                    print(f"Failed to select {category}: {e}")
                    continue  

                # Extract table data
                table = soup.find('table', class_='rk-table')
                if not table:
                    print(f"No table found for {category}")
                    continue

                universities_list = table.find_all('span', class_='tooltiptext shadow-xs')
                countries_list = table.find_all('div', class_='region-img')
                number_list = table.find_all('td', class_='')

                # Extract static data only on the first category
                if category == "Q1":
                    universities_results = [uni.text.strip() for uni in universities_list]
                    countries_results = [str(country).split('/')[-2].split('.')[0] for country in countries_list]
                    world_number_list = [' '.join(number_list[x].text.split()) for x in range(0, len(number_list), 4)]
                    total_score_results = [' '.join(number_list[x].text.split()) for x in range(2, len(number_list), 4)]

                # Extract the dynamic category scores (this will be the 'award' column)
                award_column = [' '.join(number_list[x].text.split()) for x in range(3, len(number_list), 4)]
                temp_category_scores[category] = award_column

            # Ensure all lists have the same length before appending
            num_entries = len(universities_results)
            if num_entries == 0:
                print(f"Skipping page {i-1}, no data found.")
                continue

            for key in temp_category_scores.keys():
                while len(temp_category_scores[key]) < num_entries:
                    temp_category_scores[key].append('N/A')  

            while len(total_score_results) < num_entries:
                total_score_results.append('N/A')

            universities_final += universities_results
            countries_final += countries_results
            world_number_final += world_number_list
            total_score_final += total_score_results

            for key in categories:
                category_scores_dict[key] += temp_category_scores[key]

            try:
                button_class = f"ant-pagination-item-{i}"
                button = driver.find_element(By.CLASS_NAME, button_class)
                button.click()
            except:
                print(f"Failed to navigate to page {i} for {discipline}")

        # Create a DataFrame with all categories included
        temp_df = pd.DataFrame({
            'year': [j] * len(universities_final),
            'discipline': [discipline] * len(universities_final),
            'department': [department] * len(universities_final),
            'world_rank': world_number_final,
            'institution': universities_final,
            'country_name': countries_final,
            'total_score': total_score_final,
            'Q1': category_scores_dict['Q1'],
            'CNCI': category_scores_dict['CNCI'],
            'IC': category_scores_dict['IC'],
            'TOP': category_scores_dict['TOP'],
            'AWARD_SCORE': category_scores_dict['AWARD']
        })

        shanghai_result = pd.concat([shanghai_result, temp_df], ignore_index=True)

        print(f"Data collected for {department}, saving partial results...")
        shanghai_result.to_csv(f'shanghai_results_{j}.csv', index=False) #adjust here

print("\nAll years processed. Saving final results...")
shanghai_result.to_csv('shanghai_results.csv', index=False)
print("Final results saved.")


  0%|          | 0/1 [00:00<?, ?it/s]


Processing year: 2023

Fetching data for discipline: Social Sciences, department: Management, URL: https://www.shanghairanking.com/rankings/gras/2023/RS0511
Found 17 pages for Social Sciences
Processing page 1/17 for Management

Selecting category: Q1

Selecting category: CNCI

Selecting category: IC

Selecting category: TOP

Selecting category: AWARD
Processing page 2/17 for Management

Selecting category: Q1

Selecting category: CNCI

Selecting category: IC

Selecting category: TOP

Selecting category: AWARD
Processing page 3/17 for Management

Selecting category: Q1

Selecting category: CNCI

Selecting category: IC

Selecting category: TOP

Selecting category: AWARD
Processing page 4/17 for Management

Selecting category: Q1

Selecting category: CNCI

Selecting category: IC

Selecting category: TOP

Selecting category: AWARD
Processing page 5/17 for Management

Selecting category: Q1

Selecting category: CNCI

Selecting category: IC

Selecting category: TOP

Selecting category: AWA

100%|██████████| 1/1 [00:32<00:00, 32.53s/it]

Failed to navigate to page 18 for Social Sciences
Data collected for Management, saving partial results...

All years processed. Saving final results...
Final results saved.
